# Phase 4 — HAR-RV Forecasting and HAR-Based VRP Inspection

This notebook inspects Phase 4 outputs only.

It loads:
- HAR forecast panels
- HAR-VRP panels
- forecast accuracy table
- coefficient history
- no-lookahead audit table
- metadata

It does not:
- fit HAR models
- generate production outputs
- tune parameters
- create regimes
- create trading signals
- run backtests

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


cwd = Path.cwd()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
REPORT_FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
us_har = pd.read_parquet(DATA_DIR / "us_har_forecast.parquet")
india_har = pd.read_parquet(DATA_DIR / "india_har_forecast.parquet")

us_vrp_har = pd.read_parquet(DATA_DIR / "us_vrp_har.parquet")
india_vrp_har = pd.read_parquet(DATA_DIR / "india_vrp_har.parquet")

accuracy = pd.read_csv(REPORT_TABLE_DIR / "har_forecast_accuracy.csv")
coefficients = pd.read_csv(REPORT_TABLE_DIR / "har_coefficients.csv")
audit = pd.read_csv(REPORT_TABLE_DIR / "har_no_lookahead_audit.csv")
metadata_path = REPORT_TABLE_DIR / "har_metadata.json"

print("US HAR:", us_har.shape)
print("India HAR:", india_har.shape)
print("US HAR-VRP:", us_vrp_har.shape)
print("India HAR-VRP:", india_vrp_har.shape)
print("Accuracy:", accuracy.shape)
print("Coefficients:", coefficients.shape)
print("Audit:", audit.shape)
print("Metadata exists:", metadata_path.exists())

In [ ]:
HAR_FORECAST_REQUIRED = [
    "date",
    "market",
    "target_col",
    "target_start_date",
    "target_end_date",
    "rv_gk_22d_forward_ann_label",
    "har_rv_d_lag1_ann",
    "har_rv_w_lag1_ann",
    "har_rv_m_lag1_ann",
    "naive_lagged_22d_rv_ann",
    "expanding_mean_forward_rv_baseline",
    "rolling_mean_forward_rv_baseline",
    "har_rv_gk_22d_forecast_ann",
    "har_model_name",
    "har_train_start_date",
    "har_train_end_date",
    "har_n_train",
    "har_oos_flag",
    "har_forecast_available",
    "har_blocked_reason",
]

VRP_HAR_REQUIRED = [
    "date",
    "market",
    "iv_ann",
    "har_rv_gk_22d_forecast_ann",
    "har_forecast_available",
    "har_blocked_reason",
    "vrp_har_gk",
    "vrp_har_gk_positive",
]

for name, df, required in [
    ("us_har", us_har, HAR_FORECAST_REQUIRED),
    ("india_har", india_har, HAR_FORECAST_REQUIRED),
    ("us_vrp_har", us_vrp_har, VRP_HAR_REQUIRED),
    ("india_vrp_har", india_vrp_har, VRP_HAR_REQUIRED),
]:
    missing = [col for col in required if col not in df.columns]
    print(name, "missing:", missing)
    assert not missing

In [ ]:
def forecast_status_summary(df: pd.DataFrame, market: str) -> pd.DataFrame:
    out = (
        df["har_blocked_reason"]
        .fillna("available")
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="count")
    )
    out["market"] = market
    out["share"] = out["count"] / len(df)
    return out[["market", "status", "count", "share"]]


status_summary = pd.concat(
    [
        forecast_status_summary(us_har, "US"),
        forecast_status_summary(india_har, "INDIA"),
    ],
    ignore_index=True,
)

status_summary

In [ ]:
audit_check = audit.copy()

available = audit_check[audit_check["forecast_available"].astype(bool)].copy()

available["forecast_date"] = pd.to_datetime(available["forecast_date"])
available["max_training_target_end_date"] = pd.to_datetime(
    available["max_training_target_end_date"]
)

bad = available[
    available["max_training_target_end_date"] >= available["forecast_date"]
]

print("Available audit rows:", len(available))
print("Bad audit rows:", len(bad))

assert bad.empty
assert available["rule_target_end_before_forecast_date"].astype(bool).all()

available.head()

In [ ]:
accuracy

In [ ]:
metric_cols = ["rmse", "mae", "qlike", "bias", "correlation"]

accuracy_pivot = accuracy.pivot(
    index="forecast_col",
    columns="market",
    values=metric_cols,
)

accuracy_pivot

In [ ]:
loss_metrics = ["mse", "rmse", "mae", "qlike"]

best_rows = []

for market in accuracy["market"].unique():
    sub = accuracy[accuracy["market"] == market]
    for metric in loss_metrics:
        row = sub.loc[sub[metric].idxmin()]
        best_rows.append(
            {
                "market": market,
                "metric": metric,
                "best_forecast": row["forecast_col"],
                "value": row[metric],
            }
        )

best_by_metric = pd.DataFrame(best_rows)
best_by_metric

In [ ]:
def plot_forecast_vs_target(
    df: pd.DataFrame,
    market: str,
    start: str | None = None,
    end: str | None = None,
) -> None:
    plot_df = df.copy()
    plot_df["date"] = pd.to_datetime(plot_df["date"])

    if start is not None:
        plot_df = plot_df[plot_df["date"] >= pd.to_datetime(start)]

    if end is not None:
        plot_df = plot_df[plot_df["date"] <= pd.to_datetime(end)]

    plt.figure(figsize=(14, 6))
    plt.plot(
        plot_df["date"],
        plot_df["rv_gk_22d_forward_ann_label"],
        label="Forward realised RV label",
    )
    plt.plot(
        plot_df["date"],
        plot_df["har_rv_gk_22d_forecast_ann"],
        label="HAR forecast",
    )
    plt.plot(
        plot_df["date"],
        plot_df["naive_lagged_22d_rv_ann"],
        label="Naive lagged 22d RV",
        alpha=0.7,
    )
    plt.title(f"{market}: HAR forecast vs realised forward RV")
    plt.xlabel("Date")
    plt.ylabel("Annualized variance")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
plot_forecast_vs_target(us_har, "US")

In [ ]:
plot_forecast_vs_target(india_har, "INDIA")

In [ ]:
def residual_summary(df: pd.DataFrame, market: str) -> pd.DataFrame:
    out = df.copy()
    out["residual"] = (
        out["rv_gk_22d_forward_ann_label"]
        - out["har_rv_gk_22d_forecast_ann"]
    )

    available = out[out["har_forecast_available"].astype(bool)].copy()

    return pd.DataFrame(
        [
            {
                "market": market,
                "n": available["residual"].count(),
                "mean_residual": available["residual"].mean(),
                "median_residual": available["residual"].median(),
                "std_residual": available["residual"].std(),
                "p05_residual": available["residual"].quantile(0.05),
                "p95_residual": available["residual"].quantile(0.95),
                "min_residual": available["residual"].min(),
                "max_residual": available["residual"].max(),
            }
        ]
    )


pd.concat(
    [
        residual_summary(us_har, "US"),
        residual_summary(india_har, "INDIA"),
    ],
    ignore_index=True,
)

In [ ]:
def plot_residuals(df: pd.DataFrame, market: str) -> None:
    plot_df = df.copy()
    plot_df["date"] = pd.to_datetime(plot_df["date"])
    plot_df["residual"] = (
        plot_df["rv_gk_22d_forward_ann_label"]
        - plot_df["har_rv_gk_22d_forecast_ann"]
    )

    available = plot_df[plot_df["har_forecast_available"].astype(bool)]

    plt.figure(figsize=(14, 5))
    plt.plot(available["date"], available["residual"], label="Target - HAR forecast")
    plt.axhline(0.0, linewidth=1)
    plt.title(f"{market}: HAR forecast residuals")
    plt.xlabel("Date")
    plt.ylabel("Annualized variance residual")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


plot_residuals(us_har, "US")
plot_residuals(india_har, "INDIA")

In [ ]:
coefficients.head()

In [ ]:
if "hac_available" in coefficients.columns:
    hac_summary = (
        coefficients.groupby("market")["hac_available"]
        .agg(["count", "sum", "mean"])
        .rename(columns={"sum": "hac_rows", "mean": "hac_share"})
    )
else:
    hac_summary = pd.DataFrame()

hac_summary

In [ ]:
def plot_coefficients(coef_df: pd.DataFrame, market: str) -> None:
    df = coef_df[coef_df["market"] == market].copy()
    df["date"] = pd.to_datetime(df["date"])

    coef_cols = [
        "coef_const",
        "coef_har_rv_d_lag1_ann",
        "coef_har_rv_w_lag1_ann",
        "coef_har_rv_m_lag1_ann",
    ]

    plt.figure(figsize=(14, 6))

    for col in coef_cols:
        if col in df.columns:
            plt.plot(df["date"], df[col], label=col)

    plt.axhline(0.0, linewidth=1)
    plt.title(f"{market}: expanding-window HAR coefficient paths")
    plt.xlabel("Date")
    plt.ylabel("Coefficient")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


plot_coefficients(coefficients, "US")
plot_coefficients(coefficients, "INDIA")

In [ ]:
def check_har_vrp_validity(df: pd.DataFrame, market: str) -> pd.DataFrame:
    out = df.copy()

    unavailable = ~out["har_forecast_available"].astype(bool)
    bad = unavailable & out["vrp_har_gk"].notna()

    return pd.DataFrame(
        [
            {
                "market": market,
                "rows": len(out),
                "available_forecasts": int(out["har_forecast_available"].sum()),
                "bad_unavailable_vrp_har_rows": int(bad.sum()),
                "vrp_har_non_null": int(out["vrp_har_gk"].notna().sum()),
            }
        ]
    )


pd.concat(
    [
        check_har_vrp_validity(us_vrp_har, "US"),
        check_har_vrp_validity(india_vrp_har, "INDIA"),
    ],
    ignore_index=True,
)

In [ ]:
vrp_cols = [
    "iv_ann",
    "vrp_backward_gk",
    "vrp_forward_expost_gk_label",
    "har_rv_gk_22d_forecast_ann",
    "vrp_har_gk",
]

def summarize_vrp(df: pd.DataFrame, market: str) -> pd.DataFrame:
    rows = []

    for col in vrp_cols:
        if col not in df.columns:
            continue

        values = pd.to_numeric(df[col], errors="coerce")
        rows.append(
            {
                "market": market,
                "column": col,
                "count": values.count(),
                "mean": values.mean(),
                "median": values.median(),
                "std": values.std(),
                "p05": values.quantile(0.05),
                "p95": values.quantile(0.95),
                "positive_share": (values.dropna() > 0).mean(),
            }
        )

    return pd.DataFrame(rows)


pd.concat(
    [
        summarize_vrp(us_vrp_har, "US"),
        summarize_vrp(india_vrp_har, "INDIA"),
    ],
    ignore_index=True,
)

In [ ]:
def plot_vrp_comparison(df: pd.DataFrame, market: str) -> None:
    plot_df = df.copy()
    plot_df["date"] = pd.to_datetime(plot_df["date"])

    plt.figure(figsize=(14, 6))

    if "vrp_backward_gk" in plot_df.columns:
        plt.plot(
            plot_df["date"],
            plot_df["vrp_backward_gk"],
            label="Backward VRP GK",
            alpha=0.7,
        )

    plt.plot(
        plot_df["date"],
        plot_df["vrp_har_gk"],
        label="HAR prospective VRP GK",
        alpha=0.9,
    )

    plt.axhline(0.0, linewidth=1)
    plt.title(f"{market}: backward VRP vs HAR-based prospective VRP")
    plt.xlabel("Date")
    plt.ylabel("Annualized variance spread")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


plot_vrp_comparison(us_vrp_har, "US")
plot_vrp_comparison(india_vrp_har, "INDIA")

In [ ]:
display_cols = [
    "market",
    "forecast_col",
    "n_obs",
    "rmse",
    "mae",
    "qlike",
    "bias",
    "correlation",
    "directional_accuracy_vs_baseline",
]

accuracy[display_cols].sort_values(["market", "qlike"])

## Interpretation notes

### US

HAR is compared against:
- naive lagged 22-day RV
- expanding historical forward-RV mean
- rolling historical forward-RV mean

Report only what the table supports. If HAR has lower RMSE, MAE, and QLIKE, state that it improves over baselines on those metrics.

### India

India should be interpreted separately. If HAR improves RMSE and QLIKE but not MAE or correlation, state that it improves variance-forecast loss metrics but does not dominate all naive baselines.

### No-lookahead status

The audit requires:

`max_training_target_end_date < forecast_date`

for every available forecast row.

### HAR-VRP validity

`vrp_har_gk` is only valid when:

`har_forecast_available == True`

Unavailable forecast rows must have null `vrp_har_gk`.

In [ ]:
print("Notebook inspection complete.")
print("No models were fitted in this notebook.")
print("No production outputs were written in this notebook.")